# Local Mock Experiment: Semantically Independent Single-Token Words

This notebook runs a **local mock** of token selection without the cluster.

Pipeline:
1. Build a smaller `english_mock.txt` from `english.txt`.
2. Run `scripts/sample_semantically_independent_tokens.py` with BERT.
3. Inspect selected tokens.


In [ ]:
import subprocess

res = subprocess.run(['uv', 'add', 'numpy', 'torch', 'transformers'], text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'uv add failed with code {res.returncode}')


In [ ]:
from pathlib import Path
import random

ROOT = Path('/Users/giaco/Documents/projects/NLP/hico')
ENGLISH = ROOT / 'data/english.txt'
MOCK = ROOT / 'data/english_mock.txt'
SCRIPT = ROOT / 'scripts' / 'uncorr_tkns.py'

assert SCRIPT.exists(), f'Missing script: {SCRIPT}'
assert ENGLISH.exists(), f'Missing wordlist: {ENGLISH}'

words = [w.strip().lower() for w in ENGLISH.read_text(encoding='utf-8').splitlines() if w.strip()]
random.seed(0)
random.shuffle(words)
subset = words[:4000]
MOCK.write_text('\n'.join(subset) + '\n', encoding='utf-8')
print(f'Wrote {len(subset)} words to {MOCK}')

In [ ]:
import subprocess

output_base = ROOT / 'selected_mock_bert_base.txt'
cmd = [
    'uv', 'run', 'python', str(SCRIPT),
    '--model-id', 'bert-base-uncased',
    '--wordlist', str(MOCK),
    '--layer', '12',
    '--k', '32',
    '--batch-size', '64',
    '--device', 'mps',
    '--output', str(output_base),
]

res = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Command failed with code {res.returncode}')


In [ ]:
# Optional heavier run
# If this is too slow on your laptop, skip this cell.
output_large = ROOT / 'selected_mock_bert_large.txt'
cmd = [
    'uv', 'run', 'python', str(SCRIPT),
    '--model-id', 'bert-large-uncased',
    '--wordlist', str(MOCK),
    '--layer', '24',
    '--k', '32',
    '--batch-size', '16',
    '--device', 'mps',
    '--output', str(output_large),
]

res = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f'Command failed with code {res.returncode}')


In [ ]:
# Preview output
path = ROOT / 'selected_mock_bert_base.txt'
assert path.exists(), f'Missing output file: {path}'

lines = path.read_text(encoding='utf-8').splitlines()
print(f'Rows: {len(lines)}')
for line in lines[:20]:
    print(line)

In [ ]:
import random
from typing import List, Dict, Any


def generate_grid_dataset(
    words: List[str],
    context_len: int,
    seed: int = 0,
    split_context_across_batch: bool = False,  # True => seq_len = context_len // 16
) -> Dict[str, Any]:
    assert len(words) == 16, "Need exactly 16 words for a 4x4 grid."
    assert context_len > 0

    rng = random.Random(seed)

    # 1) Randomly place words on a 4x4 grid
    placed = words[:]
    rng.shuffle(placed)
    grid = [placed[r * 4:(r + 1) * 4] for r in range(4)]

    # 2) Token ids (0..15) in grid-placement order
    id_to_word = {i: w for i, w in enumerate(placed)}
    word_to_id = {w: i for i, w in id_to_word.items()}

    # 3) 4-neighbor adjacency
    pos_of = {word_to_id[grid[r][c]]: (r, c) for r in range(4) for c in range(4)}
    id_at = {(r, c): word_to_id[grid[r][c]] for r in range(4) for c in range(4)}
    neighbors = {i: [] for i in range(16)}
    for i in range(16):
        r, c = pos_of[i]
        for rr, cc in [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)]:
            if 0 <= rr < 4 and 0 <= cc < 4:
                neighbors[i].append(id_at[(rr, cc)])

    # 4) Batch settings
    batch_size = 16  # number of nodes
    seq_len = context_len // batch_size if split_context_across_batch else context_len
    seq_len = max(seq_len, 1)

    # 5) One sequence per start node
    batch = []
    for start_id in range(16):
        cur = start_id
        seq = []
        for _ in range(seq_len):
            seq.append({
                "token_id": cur,       # required by your downstream code
                "word": id_to_word[cur]
            })
            cur = rng.choice(neighbors[cur])
        batch.append(seq)

    return {
        "grid_words": grid,                 # 4x4 layout
        "word_to_id": word_to_id,
        "id_to_word": id_to_word,
        "neighbors": neighbors,
        "batch_size": batch_size,           # 16
        "sequence_length": seq_len,         # context_len OR context_len//16
        "batch": batch                      # len(batch)=16, len(batch[i])=seq_len
    }


In [ ]:
words = ["apple","bird","car","egg","house","milk","plane","opera",
         "box","sand","sun","mango","rock","math","code","phone"]

data = generate_grid_dataset(words, context_len=1400, seed=0, split_context_across_batch=False)
# len(data["batch"]) == 16
# len(data["batch"][0]) == 1400


In [ ]:
data